# Model Comparison

The below setup is the same as notebook 02, the cleaning and data split are the same too. All models are compared on the same data, in order to make sure the comparison between models is fair and any difference in results come from the model itself.

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
df = pd.read_excel("../data/default of credit card clients.xls", header=1)
df.shape

In [ ]:
df = df.drop(columns="ID")
df = df.drop_duplicates()
df.shape

In [ ]:
df["EDUCATION"] = df["EDUCATION"].replace({0 :4, 5 :4, 6 :4})
print(df["EDUCATION"].value_counts().sort_index())

df["MARRIAGE"] = df["MARRIAGE"].replace({0 :3})
print(df["MARRIAGE"].value_counts().sort_index())

In [ ]:
df = df.rename(columns={"default payment next month": "default"})

In [ ]:
X = df.drop(columns="default")
y = df["default"]
print(X.shape, y.shape)

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

In [ ]:
print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]
num_cols = [col for col in X.columns if col not in cat_cols]
print(len(cat_cols), len(num_cols))

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

In [ ]:
def evaluate(model, X, y):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    cost = 5 * fn + 1 * fp
    return {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, zero_division = 0),
        "Recall": recall_score(y, y_pred),
        "F1": f1_score(y, y_pred, zero_division = 0),
        "PR-AUC": average_precision_score(y, y_proba),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Cost": cost
    }


This evaluation function returns all the metrics from accuracy to TP, along with cost, for any given model. I used this to make sure every model is scored the same way on validation set.

In [ ]:
baseline = Pipeline([
    ("prep", preprocessor),
    ("model", DummyClassifier(strategy = "most_frequent")),
])

baseline.fit(X_train, y_train)

In [ ]:
evaluate(baseline, X_val, y_val)

These results are the exact same as notebook 02's baseline results, just to confirm the evaluate function works correctly, with both costs being 6630.

In [ ]:
log_reg = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(class_weight = "balanced", max_iter = 1000)),
])

log_reg.fit(X_train, y_train)

In [ ]:
evaluate(log_reg, X_val, y_val)

Above, I used a Logistic Regression model and set the class weight to balanced, so that in this model not catching a defaulter is more costly, which pushes the model to try catch defaulters. I set max_iter to 1000 just to give it enough iterations to finish training. Compared to the baseline, it's accuracy dropped 0.78 to 0.69, recall rose 0 to 0.65, and cost went down from 6630 to 3727. PR-AUC rose from 0.22 to 0.5. To add context to those figures, this model now catches about 65% of all defaulters. Even though the accuracy is lower for this model, it's still a better model, because it actually catches a large amount of true defaulters compared to the lazy model, and it reduces the cost by almost a half. The higher PR-AUC means this model is better at keeping a higher precision while still keeping its recall. This suggests this model performs better than the baseline lazy model.

In [ ]:
tree = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeClassifier(class_weight = "balanced", max_depth = 5, random_state = 42))
])

tree.fit(X_train, y_train)

In [ ]:
evaluate(tree, X_val, y_val)

Above is the decision tree model. Its class weight is balanced as missing a defaulter costs more and max depth set to 5 to avoid overfitting, as the model would memorise noise in the training data and look great on it, but poor on the validation. The depth set to 5 keeps it simple enough to generalise. Random state = 42 so that the tree will always give the same result everytime its run. Comparing it then to the logistic regression model, it performs better, but not by much. The cost is lower at 3433, and it catches 29 more defaulters.

In [ ]:
forest = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(n_estimators = 200, max_depth = 10, class_weight = "balanced", random_state = 42, n_jobs = -1))
])

forest.fit(X_train, y_train)

In [ ]:
evaluate(forest, X_val, y_val)

This is the random forest model. I used 200 trees as each tree is trained on slightly different data and each makes its own mistakes, that get averaged out across a high number of trees. The class weights are balanced to make not catching a defaulter more costly. The depth was set to 10 so that it makes sure we can generalise it, like the last model. The depth is higher due to more decision trees, as the mistakes get averaged out so that the trees can be big without the forest overfitting. And again, random_state = 42 to keep reproducibility. This model has the best PR-AUC and F1 but a higher cost, which is because it missed more defaulters (547 missed vs 440), which is more costly than flagging a good customer. That's why the cost is higher, and I'll use threshold tuning to fix it later on.

In [ ]:
results = pd.DataFrame({
    "Baseline": evaluate(baseline, X_val, y_val),
    "Logistic Regression": evaluate(log_reg, X_val, y_val),
    "Tree": evaluate(tree, X_val, y_val),
    "Forest": evaluate(forest, X_val, y_val)    
})

In [ ]:
results

## Results Table

In [ ]:
results.T.round(3)

The above table shows all results compared against each other, being the baseline and the 3 models I used. The model with the lowest cost, is the Tree at 3433, and the model with the best PR-AUC is the forest with 0.560 but it missed more defaulters(547). Missing a defaulter is more costly than flagging a non defaulter, which increases the cost of the model. Tree also has the highest recall, being 0.668, meaning it catches the biggest proportion of the total defaulters. All 3 of these models clearly beat the baseline.

In [ ]:
forest_proba = forest.predict_proba(X_val)[:, 1]

In [ ]:
thresholds = np.arange(0.05, 1, 0.05)

In [ ]:
costs = []
for t in thresholds:
    pred = (forest_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    costs.append(5 * fn + 1 * fp)

costs

In [ ]:
best_t = thresholds[np.argmin(costs)]
best_t

In [ ]:
costs[np.argmin(costs)]

I used threshold tuning on the forest model to see could I get a better threshold to reduce cost. Before, it was set to 0.5, but if a customer had a 0.4 probability of defaulting say, they would not get flagged as a defaulter. If they defaulted and were missed by the model, it would cost 5x as much. If they were flagged and didn't default, it would not cost as much as not picking them up. So from this, we try to find the best threshold. I did this by making a loop, testing thresholds from 0.05 to 0.95, in steps of 0.05, on the validation set, so that the test set stays untouched for my final evaluation. This way, I was able to see the lowest cost from those thresholds. As a result, I got 0.35, which had lowered the cost from 3509 to 3318, the new lowest cost for all models so far(threshold yet to be applied to others!). This optimal threshold has now saved more cost than the default 0.5, and I plan to apply this to the other models too to make things fair for comparison.

# Tuning Tree and Logistic Regression next

In [ ]:
def tune_thresholding(model, X, y):
    proba = model.predict_proba(X)[:, 1]
    costs = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
        costs.append(5 * fn + 1 * fp)
    best = np.argmin(costs)
    return thresholds[best], costs[best]

In [ ]:
print("Logistic Regression:", tune_thresholding(log_reg, X_val, y_val))
print("Tree:", tune_thresholding(tree, X_val, y_val))
print("Forest:", tune_thresholding(forest, X_val, y_val))

That was the tuning applied to all models. I created a function to apply the thresholding to all models to keep things fair, using the same tuning for each model. The results from the 3 models show the forest is still the cheapest so far with the cost being the lowest at 3318. The Log Reg model didn't change, with a threshold still at 0.5, mainly because the class weight = balanced had already shifted the model in the right direction. The tree improved, but barely (3433 to 3413), with a threshold at 0.4. This would be because the depth of 5 only gives a limited number of probabilities so moving the threshold won't change a whole lot. However, the threshold was chosen and the cost was measured on the same validation set, so the cheapest result may be too optimistic, so the real test will be applying the threshold to the test set at the end

In [ ]:
best_pred = (forest_proba >= best_t).astype(int)
ConfusionMatrixDisplay.from_predictions(y_val, best_pred, display_labels = ["No default", "Default"])
plt.title("Random Forest - Confusion Matrix (threshold of 0.35)")
plt.show()

This confusion matrix shows the forests predictions on the validation set using 0.35 as it's threshold. Of all 1326 defaulters, it caught 1059 and missed 267. That recall is around 0.80(catching 80% of defaulters), which is up from 0.59 at a 0.5 threshold. 267 box is the FN and 1983 is FP. This model with 0.35 flagged 1983 false alarms vs 774 at threshold 0.5, but the 0.5 threshold missed 547 defaulters whereas 0.35 threshold only missed 267, lowering the cost. This shows why this model at a lower threshold is cheaper than the same at 0.5, as it saves cost due to FN being worth 5x more than FP.

In [ ]:
val = X_val.copy()
val["actual"] = y_val
val["pred"] = best_pred

defaulters = val[val["actual"] == 1]
defaulters.groupby("pred")[["PAY_0", "LIMIT_BAL", "PAY_AMT1"]].mean().round(1)

Next is error analysis, where I compared missed defaulters vs caught ones. The ones that defaulted and got missed, were ones that had paid on time last month, had bigger payments and higher credit limits, so they weren't flagged. This shows the model can struggle with finding defaulters if they appear to be good customers up until the moment they default.

## Conclusion

In conclusion, the best model so far is the forest model, set to a threshold of 0.35. It had the lowest cost at 3318 vs the other models at 3413, 3727, and baseline 6630. Forest model also had the best PR-AUC at 0.56, and it has recall at 0.80 (catches 80% of defaulters). However, a couple of limitations of this model are, it assumes the cost of missing a defaulter is 5x more than flagging a good one, this is just an assumption and not based on fact. The model misses defaulters who look like they're good customers too, which isn't ideal. Another is that it flags lots of false alarms (1983), which could be bad for the customer morale of the bank. Next, I will test the final model on the test set one time, and might then look at fairness and feature importance.